# Training

How the twenty runs behind this project's results were produced, and how to check
they were healthy.

Training itself needs a GPU and the prepared cache. **Everything from section 4
down runs on a fresh clone with neither** — `experiments_log/` is tracked, so the
hyper-parameters and per-epoch curves of all twenty runs are already in the repo.

Results are read in `MSN_compare_runs.ipynb`; this notebook stops at "did the runs
come out clean".

## 1 · Setup

In [ ]:
import json
import os
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
sys.path.insert(0, os.path.join(REPO, "src", "models"))

import numpy as np
import pandas as pd
from run_kfold import CONFIGS, N_FOLDS          # the experiment grid, defined once

LOG = os.path.join(REPO, "experiments_log")
print(f"{len(CONFIGS)} configs x {N_FOLDS} folds = {len(CONFIGS) * N_FOLDS} runs")

## 2 · The 2x2

Two things vary: whether the loss carries the density-aware Chamfer term (DCD),
and whether it carries an explicit repulsion term. Everything else is identical.

```
                  no repulsion        repulsion
     DCD          lr_fix_only         rep_w05
     no DCD       cd_only             cd_rep05_full
```

The names are historical and do not describe the cells — `lr_fix_only` was once a
control for a learning-rate fix. Renaming them would break the merge keys in
`eval_all_runs.csv`, which this project has already lost data to once.

In [ ]:
for name, flags in CONFIGS.items():
    print(f"{name:16} {' '.join(flags)}")

## 3 · Launching the sweep

One model at a time, five folds each. `run_kfold.py` is a driver around
`train_skullfix.py`; it launches each fold as a fresh subprocess.

```bash
tmux new -s kfold                                  # survives a dropped connection
$PY src/models/run_kfold.py --dry-run cd_only      # print the plan first
$PY src/models/run_kfold.py cd_only                # ~3 h for five folds
$PY src/models/run_kfold.py lr_fix_only
$PY src/models/run_kfold.py rep_w05
$PY src/models/run_kfold.py cd_rep05_full
```

Roughly 12 hours total, 14.3 GB of checkpoints. Re-running a command skips folds
that already finished, so an interruption costs at most the fold in flight.

⚠️ **Do not edit `src/models/` or `data/` while a sweep runs.** Each fold is a new
subprocess, so an edit lands on the next fold and the five stop being one
experiment.

## 4 · What the driver guards

A bare `for` loop over folds fails in five ways this project has hit:

| Guard | The failure it prevents |
|---|---|
| Abort on hitting `--epochs` | A truncated run was still descending, so its "best" is not a result. A loop would carry on and you notice hours later |
| Check free disk first | Twenty checkpoints need 14.3 GB; running out mid-sweep looks like a training bug |
| Skip finished folds | Makes recovery "re-run the same command" instead of a decision |
| Self-check and archive per fold | Otherwise it is twenty manual steps, and the 3 a.m. one gets skipped |
| Warn but continue on soft signals | Few LR drops, a noisy tail, a high val/train ratio mean *read carefully*, not *broken* |

Three traps that silently corrupt a sweep rather than stopping it:

- **`--from-run` must not be used.** It replays an old run's hyper-parameters,
  including `n_folds=0`, quietly turning a fold into a single-split run.
- **`--dcd-lambda 2` must be explicit.** The default is 1, and the existing DCD
  cells use 2; omitting it makes the sweep incomparable with them.
- **Run names must be `<config>_f<fold>`.** `report.fold_frame` reads the fold
  from `run.json` and cross-checks the name, which is the only defence against a
  typo in twenty hand-written names.

## 5 · Were the runs clean?

All read from the archived records — no GPU, no data, no weights.

**One hard condition:** a run is quotable only if **EarlyStopping** stopped it.
Hitting the epoch ceiling means it was still improving when cut off, which makes
its "best" an artefact of where the ceiling was.

**Three advisory ones**, which the driver logs but does not stop on (`--strict`
changes that): enough LR decay to have annealed, a settled tail, and a val/train
ratio that is not drifting. They mean *read this run carefully*, not *discard it*.

In [ ]:
HARD = "EarlyStopping"
SOFT = {"lr_drops": (">=", 5), "tail30_std": ("<", 0.02), "val/train": ("<", 1.25)}

rows = []
for cfg in CONFIGS:
    for f in range(N_FOLDS):
        run = f"{cfg}_f{f}"
        meta = json.load(open(os.path.join(LOG, run, "run.json")))
        h = pd.read_csv(os.path.join(LOG, run, "history.csv"))

        n = len(h)
        best = int(h["val_loss"].idxmin()) + 1
        if n - best == meta["early_stop_patience"]:
            stop = HARD
        elif n >= meta["epochs"]:
            stop = "CEILING -- discard"
        else:
            stop = "wall clock"

        rows.append({"run": run, "epochs": n, "best": best, "stop": stop,
                     "lr_drops": int((h["lr"].diff() < -1e-12).sum()),
                     "tail30_std": round(float((h["val_cd_t_metric"] * meta["scale_mm"]).tail(30).std()), 4),
                     "val/train": round(float(h["val_cd_t_metric"][best - 1] / h["cd_t_metric"][best - 1]), 3),
                     "CD_t_mm": round(meta["best_val_cd_t_mm"], 3)})

runs = pd.DataFrame(rows)
print(runs.to_string(index=False))

assert (runs["stop"] == HARD).all(), runs[runs["stop"] != HARD]
print(f"\nHard condition: {len(runs)}/{len(runs)} stopped on EarlyStopping, none hit the ceiling.")

for col, (op, lim) in SOFT.items():
    bad = runs[runs[col] >= lim] if op == "<" else runs[runs[col] < lim]
    verdict = "clear" if bad.empty else ", ".join(f"{r.run} ({r[col]})" for _, r in bad.iterrows())
    print(f"  {col:12} {op} {lim:<6} {verdict}")

⭐ One advisory trips: **`rep_w05_f0` has a tail standard deviation of 0.039**,
about twice the 0.02 guideline. Its validation curve was still wobbling when
EarlyStopping fired, so its best epoch is a less settled reading than the other
nineteen.

Left in rather than re-run, having checked what dropping it would do. Fold 0 is
`rep_w05`'s worst fold, and on the "+repulsion, with DCD" edge it contributes the
smallest per-fold effect by far (-0.001 against -0.053 to -0.086). Excluding it
would move that edge from -0.052 to -0.065 and halve its spread — i.e. it would
make this project's own claim look stronger. On the edge that came out null,
excluding it moves the mean from +0.016 to -0.018 and leaves the spread
essentially unchanged, so the null is not an artefact of this fold either way.

Dropping a run because keeping it is inconvenient is how a sweep stops meaning
anything.

⚠️ `CD_t_mm` above is the training-time figure from `run.json`: best epoch,
dataset-average scale. It reads about 0.09 mm below the per-skull figure the
thesis quotes from `eval_all_runs.csv`. Use it to judge whether a run behaved,
never to compare configurations.

In [ ]:
summary = runs.assign(config=runs.run.str.rsplit("_f", n=1).str[0])
print(summary.groupby("config", sort=False).agg(
    epochs_min=("epochs", "min"), epochs_max=("epochs", "max"),
    CD_t_mean=("CD_t_mm", "mean"), CD_t_std=("CD_t_mm", "std")).round(3).to_string())

## 6 · Training curves

Each config's five folds, so the spread between folds is visible rather than
averaged away. Dashed lines mark learning-rate drops — a run with none never
annealed, and four early runs were discarded for exactly that.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=len(CONFIGS), shared_yaxes=True,
                    subplot_titles=list(CONFIGS))
for col, cfg in enumerate(CONFIGS, start=1):
    for f in range(N_FOLDS):
        meta = json.load(open(os.path.join(LOG, f"{cfg}_f{f}", "run.json")))
        h = pd.read_csv(os.path.join(LOG, f"{cfg}_f{f}", "history.csv"))
        fig.add_scatter(x=np.arange(1, len(h) + 1), y=h["val_cd_t_metric"] * meta["scale_mm"],
                        name=f"fold {f}", legendgroup=f"f{f}", showlegend=(col == 1),
                        line=dict(width=1), row=1, col=col)

fig.update_yaxes(title_text="val CD_t (mm)", range=[6.0, 9.0], row=1, col=1)
fig.update_xaxes(title_text="epoch")
fig.update_layout(height=380, margin=dict(l=60, r=20, b=50, t=50),
                  title="Validation CD_t, five folds per config").show()

## Appendix · What differs from the released implementation

Full detail in `src/models/msn_skullfix.py`'s module docstring.

| | Change | Why |
|---|---|---|
| **A** | One similarity transform, derived from the defective cloud, applied to both clouds | Normalising each separately puts a matched pair in different frames — measured 3.6% centroid shift and a 32% inflation of the ground-truth-to-input distance |
| **B** | Distance matrix via `\|a\|^2 - 2a.b + \|b\|^2` | The tiled form needs a 12 GB tensor at batch 8 / 6144 points. After this the 187M model trains at 372 ms/step in 15.5 GiB on one 4090 |
| **C** | Default loss `cd_dcd`, not DCD alone | DCD is bounded [0,2]; at random initialisation both factors vanish and the loss pins at 1.9995 |
| **D** | Stateless seeded sampling at inference | The stateful sampler gives a different output for the same input on every call, so metrics are not reproducible |
| **E** | Other | lr 1e-7 -> 3e-4 with warmup; batch 8 -> 4 for memory; explicit split by skull id (Keras slices the tail *before* shuffling, which leaks); BERT precomputed, since one class means its output is constant; `.h5` checkpoints, as the new format stores optimiser state too (2.25 GB); voxel spacing applied as a matmul from the nrrd header |